# Lab 09: Extracting Data from PDFs

*INFO 4617/5617 · Week 9 · Chapter 9, [Extracting Data from PDFs](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html)*

This week every class is live coding: one lab in three parts, one a day. Each step says what we're building and why. Its code cells hold a comment saying what to write, and we write the code together. Cells that start with `# Given:` are already written: we read them together and run them. Nothing here is turned in. After each step, a **💬 Discuss** box holds questions to talk over; click it to open it.

We take the City of Boulder's PDFs apart the way the chapter does:
- **Part 1, Monday:** what a PDF holds. One revenue report, by hand: its pages and cover with pypdf, where one word sits with pdfplumber, and which pages are born-digital, scanned, or scanned with a text layer.
- **Part 2, Wednesday:** text and tables. Council minutes read three ways, repaired and searched; then the report's summary table, which pdfplumber finds one row short, and how to get the row back.
- **Part 3, Friday:** OCR and a time series. One row of a picture of a table, then the whole picture; then eight years of sales and use tax, from seven reports.

**The files** are the chapter's copies of the city's records, kept in the book's repository, so that a whole class doesn't hit the city's server at once. Only Step 6 asks the city's portal itself, and in class only the instructor runs it.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. Step 0 imports pypdf, pdfplumber, and pytesseract; if one stops with `ModuleNotFoundError`, run `%conda install -y -c conda-forge pypdf pdfplumber pytesseract tesseract` once in a new cell, as the chapter's [Installing pypdf, pdfplumber, and pytesseract](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#installing-pypdf-pdfplumber-and-pytesseract) says, then restart the kernel.

On Wednesday and Friday, start at that day's setup step.

# Part 1 — Monday: what a PDF holds

A PDF looks like a page and stores drawing instructions. Today we take one of the city's revenue reports and ask it, by hand, what it holds: how many pages and what the cover says (pypdf), where one word sits (pdfplumber), and which pages have text to extract at all.

## Step 0 — Set up (3 minutes)

Put your email address in the User-Agent, and run the given cell. It imports what reads PDFs, and prints the libraries' versions and Tesseract's: `pytesseract` runs a separate program, Tesseract, and a missing program shows up here as `TesseractNotFoundError`. `BOOK_DATA` is the book's folder of saved files on GitHub, and `PDF_DIR` a folder, `pdfs`, to keep our copies in. The chapter's [This Chapter's Files](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#this-chapters-files) has the code.

In [ ]:
# Given: Part 1's setup. Put your email address in HEADERS, then run it
import requests
from pathlib import Path
import pandas as pd
import pdfplumber
import pypdf
import pytesseract
from pypdf import PdfReader

print("pypdf", pypdf.__version__, "| pdfplumber", pdfplumber.__version__)
print("Tesseract", pytesseract.get_tesseract_version())

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
BOOK_DATA = "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-09/"
PDF_DIR = Path("pdfs")
PDF_DIR.mkdir(exist_ok=True)

<details>
<summary>💬 Discuss</summary>

- In week 2 we put contact information in the User-Agent. Who reads this one, now that the request goes to GitHub rather than to the city?
- Why does the book serve its own copies, instead of sending a class of forty to the city's records portal?

</details>

## Step 1 — One file, by hand (5 minutes)

Download the city's December 2024 revenue report. A PDF is bytes, not text, so keep [`response.content`](https://requests.readthedocs.io/en/stable/user/quickstart/#binary-response-content), not `response.text`. Look before you save: every PDF starts with the five bytes `%PDF-`, and GitHub answers a misspelled name with status 404 and the words `404: Not Found`. Write the bytes to a file with [`write_bytes()`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.write_bytes). The given cell gathers these lines into the chapter's `get_saved_copy(name)`, a *function*: a named block of lines that runs, with `name` filled in, each time you call it.

In [ ]:
# report_response: request BOOK_DATA + "revenue-report-2024-12.pdf", with headers=HEADERS
# Your code here

In [ ]:
# Look first: print the status code, and the first 5 bytes of report_response.content
# Your code here

In [ ]:
# report_2024: PDF_DIR / "revenue-report-2024-12.pdf"; write report_response.content to it
# Your code here

In [ ]:
# Given: Step 1's lines, gathered in the chapter's function. It downloads a file only if it
# isn't in pdfs yet, and pauses a second after each download, so a loop of them is polite
import time

def get_saved_copy(name):
    path = PDF_DIR / name
    if not path.exists():
        response = requests.get(BOOK_DATA + name, headers=HEADERS)
        path.write_bytes(response.content)
        time.sleep(1)
    return path

<details>
<summary>💬 Discuss</summary>

- Typed with a misspelled name, the request gets status 404, and `write_bytes()` saves the 14 bytes `404: Not Found` as a `.pdf`. Which line warned you, and what would `PdfReader` say when it opened the file?
- Week 6's exercises cached a page's raw HTML before parsing it, and `get_saved_copy()` does the same. If a download had saved an error page, what would every rerun read?

</details>

## Step 2 — pypdf: how many pages, and what's on the cover? (6 minutes)

Open `revenue-report-2024-12.pdf` in a PDF viewer first (double-click it in Jupyter's file browser, in the `pdfs` folder). How many pages does it have? What does the cover say? Now ask pypdf. [`PdfReader`](https://pypdf.readthedocs.io/en/stable/modules/PdfReader.html) reads the file; its `pages` is a list, counted from 0. [`extract_text()`](https://pypdf.readthedocs.io/en/stable/user/extract-text.html) writes out a page's characters in the order the page draws them, and `[:80]` keeps the first 80. Then read what the file says about itself, its [`metadata`](https://pypdf.readthedocs.io/en/stable/user/metadata.html).

In [ ]:
# reader: PdfReader(report_2024); print how many pages it has, len(reader.pages)
# Your code here

In [ ]:
# cover_text: the first page's extract_text(); print its first 80 characters
# Your code here

In [ ]:
# Look at its first 20 characters as Python stores them: "\n" is a new line
# Your code here

In [ ]:
# meta: reader.metadata; print its title, its subject, and its creator
# Your code here

In [ ]:
# Given: is the file "tagged", with a structure tree for screen readers? And when was it made?
print("tagged:", "/StructTreeRoot" in reader.trailer["/Root"])
print("created:", meta.creation_date)

<details>
<summary>💬 Discuss</summary>

- Did pypdf count the pages you counted? Before "REVENUE REPORT", the cover's text has six runs of a space and a new line. What on the page could have drawn them?
- The subject of a report on December 2024 says March 2018. What else in a file's metadata would you hesitate to rely on, and what is the creator still good for?
- Week 4's XML and JSON store their structure in the file. What does a PDF store instead, and what does that cost the person who wants its data?

</details>

## Step 3 — pdfplumber: where one word sits (7 minutes)

A PDF page is a list of drawing instructions: characters at coordinates, lines and rectangles, and images. Nothing in it says "word", "line", or "table"; every tool infers those from position. [pdfplumber](https://github.com/jsvine/pdfplumber#the-pdfplumberpage-class) gives each object's position in *points*, 1/72 of an inch, from the page's left edge (`x0`) and its `top`. Page 2 is US Letter, 8.5 by 11 inches: how many points is that? [`extract_words()`](https://github.com/jsvine/pdfplumber#extracting-text) groups characters into words, and `search()` finds a phrase. See [What's Inside a PDF](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#whats-inside-a-pdf).

In [ ]:
# report_pdf: pdfplumber.open(report_2024); summary_page: its pages[1], the second page
# Your code here

In [ ]:
# Print the page's width and its height, in points
# Your code here

In [ ]:
# words: summary_page.extract_words(); print the first word's "text" and its "x0"
# Your code here

In [ ]:
# Look at the page's first character, summary_page.chars[0]: its text, its font, its position
# Your code here

In [ ]:
# Given: the page's first eight characters, one a line, and how many of each kind of object it draws.
# round(x, 1) rounds a number to one decimal place
for char in summary_page.chars[:8]:
    print(repr(char["text"]), round(char["x0"], 1), round(char["top"], 1), char["fontname"])
print(len(summary_page.chars), "characters,", len(summary_page.rects), "rectangles,",
      len(summary_page.lines), "lines,", len(summary_page.images), "images")

In [ ]:
# found: summary_page.search("Total Sales and Use Tax"), a list of matches;
# print how many matches, and the first one's "top"
# Your code here

<details>
<summary>💬 Discuss</summary>

- The page's first character is a space, drawn above the title, and "City" comes one character at a time. What does a tool have to rebuild before it can give you one word, or one cell of a table?
- The totals row's top is about 380 points down a page 792 points tall. Find it on the page. On Wednesday, a table finder will stop just above it.
- Typed as `"total sales and use tax"`, the search finds nothing, and `found[0]` stops with an `IndexError`. Which of the two lines would you look at first?

</details>

## Step 4 — What kind of PDF is it? (9 minutes)

The right tool depends on how each page was made. **Born-digital** pages have characters to extract. **Scanned** pages are one image and no characters: they need *OCR*, optical character recognition. **Scanned with a text layer** pages are one image plus text from someone else's OCR, to test before you trust it. Count one page by hand: its characters and its [`images`](https://pypdf.readthedocs.io/en/stable/user/extract-images.html). A given loop then counts every page of three files; a `for` loop runs its indented lines once for each item. See [What Kind of PDF Is It?](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#what-kind-of-pdf-is-it).

In [ ]:
# last_page: reader.pages[14], the last page; print len() of its extract_text(), and of its images
# Your code here

In [ ]:
# Given: every page's characters and images, for three files. .append() adds to a list
for name in ["revenue-report-2024-12.pdf", "revenue-report-2018.pdf", "council-minutes-2018-12-04.pdf"]:
    characters = []
    images = []
    for page in PdfReader(get_saved_copy(name)).pages:
        characters.append(len(page.extract_text()))
        images.append(len(page.images))
    print(name)
    print("  characters:", characters)
    print("  images:    ", images)

In [ ]:
# minutes_2018: PdfReader(get_saved_copy("council-minutes-2018-12-04.pdf")); print its creator
# Your code here

In [ ]:
# first_2018: its first page's extract_text(); print the first 60 characters
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

A short version of Additional Exercise 5's quality check.

**Do:** set `sprint_file` to one of two files the chapter mentions but doesn't profile, `"council-minutes-2000-12-19.pdf"` or `"revenue-report-2021-12.pdf"`, and run the given cell. It gives each page a route from the chapter's table: born-digital (no image), OCR (an image and under 100 characters), or test the text layer (an image and more text).
**Share:** your file, how many pages take each route, and which pages you'd send to OCR.

In [ ]:
# Microsprint: sprint_file: "council-minutes-2000-12-19.pdf" or "revenue-report-2021-12.pdf"
# Your code here

In [ ]:
# Given: each page of sprint_file: its number (from 1), its characters, and its route.
# "elif" (else if) is checked only when the "if" above it wasn't true
number = 0
for page in PdfReader(get_saved_copy(sprint_file)).pages:
    number = number + 1
    characters = len(page.extract_text())
    if len(page.images) == 0:
        print("page", number, characters, "characters: born-digital")
    elif characters < 100:
        print("page", number, characters, "characters: OCR")
    else:
        print("page", number, characters, "characters: test the text layer")

<details>
<summary>💬 Discuss</summary>

- The report's last page has an image and 265 characters, the note above a picture of a table. Would the microsprint's 100-character rule send it to OCR? The chapter says no threshold tells such pages apart: what does?
- The scanner's own OCR turned "COUNCIL" into "COTJNCIL". Why is a text layer you didn't make something to test rather than something to trust?
- How many pages of your file go to OCR? Would one character count for the whole file have told you which ones?

</details>

## End of Part 1

On Wednesday, Part 2 reads text three ways and repairs it, then finds the report's summary table, and the row the table finder leaves out. It starts with its own setup step, so you don't need today's cells to have run.

# Part 2 — Wednesday: text and tables

Today we read text and tables out of born-digital pages. First, where the files came from (the instructor runs it). Then the December 5, 2024 council minutes, three ways; a repair by hand before the chapter's four; the report's summary table, which pdfplumber finds one row short; and the checks that tell us so.

## Step 5 — Set up again (3 minutes)

Part 2 starts fresh, so a student who missed Monday can start here. Put your email address in the User-Agent, and run the two given cells. The second holds Monday's `get_saved_copy()`, which downloads a file only if it isn't in `pdfs` yet, and opens the report's summary page, page 2, as Monday's Step 3 did.

In [ ]:
# Given: Part 2's setup. Put your email address in HEADERS, then run it
import re
import time
import requests
from pathlib import Path
import pandas as pd
import pdfplumber
from pypdf import PdfReader

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
BOOK_DATA = "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-09/"
PDF_DIR = Path("pdfs")
PDF_DIR.mkdir(exist_ok=True)

In [ ]:
# Given: Monday's get_saved_copy(), the December 2024 report, and its summary page
def get_saved_copy(name):
    path = PDF_DIR / name
    if not path.exists():
        response = requests.get(BOOK_DATA + name, headers=HEADERS)
        path.write_bytes(response.content)
        time.sleep(1)
    return path

report_2024 = get_saved_copy("revenue-report-2024-12.pdf")
report_pdf = pdfplumber.open(report_2024)
summary_page = report_pdf.pages[1]

<details>
<summary>💬 Discuss</summary>

- Monday's typed lines are in these cells now. Which of them would you change first to read another city's files?

</details>

## Step 6 — Where the files came from: the city's portal (5 minutes)

**In class, only the instructor runs this step.** On October 6, 2026, the city's server stopped answering after 13 to 17 requests in a row. Write the cells now, and run them at home.

The book's copies came from the city's Laserfiche WebLink portal, which lists a folder when it gets a [POST](https://requests.readthedocs.io/en/stable/user/quickstart/#more-complicated-post-requests) of the folder's number as JSON; the browser's Network tab shows it. The server leaves out a certificate, so pass the book's bundle, [`verify=bundle`](https://requests.readthedocs.io/en/stable/user/advanced/#ssl-cert-verification), never `verify=False`. Look at the status and content type before reading JSON: some errors come back as plain text. See [Getting the Files](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#getting-the-files).

In [ ]:
# Given: in class, only the instructor runs this step. The book's certificate bundle, and its
# manifest of where each copy came from; the portal's listing address, and the body for one folder
import hashlib

bundle = get_saved_copy("boulder-ca-bundle.pem")
manifest = pd.read_csv(get_saved_copy("manifest.csv"))
PORTAL = "https://documents.bouldercolorado.gov/WebLink/"
LISTING = PORTAL + "FolderListingService.aspx/GetFolderListing2"
folder_body = {"repoName": "LF8PROD2", "folderId": 194345, "getNewListing": True,
               "start": 0, "end": 100, "sortColumn": "", "sortAscending": True}

In [ ]:
# folder_response: POST to LISTING with json=folder_body, headers=HEADERS, and verify=bundle
# Your code here

In [ ]:
# Look first: print the status code, and the "Content-Type" of its headers
# Your code here

In [ ]:
# listing: folder_response.json(); folder: listing["data"]; print folder["path"]
# Your code here

In [ ]:
# Given: each entry in the folder: a folder of its own (type 0), or a report and its page count
for entry in folder["results"]:
    if entry["type"] == 0:
        print(entry["entryId"], entry["name"], "(a folder)")
    else:
        print(entry["entryId"], entry["name"], "(" + str(entry["data"][1]) + " pages)")

In [ ]:
# Given: after the chapter's 30-second pause, download the December 2024 report (docid 194397), and
# check that it is the book's copy: the SHA-256 hash of its bytes against the manifest's
time.sleep(30)
file_params = {"docid": 194397, "dbid": 0, "repo": "LF8PROD2"}
file_response = requests.get(PORTAL + "ElectronicFile.aspx", params=file_params,
                             headers=HEADERS, verify=bundle)
print(file_response.headers["Content-Type"], len(file_response.content), file_response.content[:5])
record = manifest[manifest["docid"] == 194397].iloc[0]
print(record["file"], "retrieved", record["retrieved_utc"])
print("the book's copy:", hashlib.sha256(file_response.content).hexdigest() == record["sha256"])

<details>
<summary>💬 Discuss</summary>

- Week 8 found hidden APIs in the Network tab. Which parts of this request (the folder number, `LF8PROD2`, the body) would break first if the city upgraded its portal?
- Python's default check fails on this server, and the book's bundle passes. Why is `verify=False` the wrong fix, even for public records?
- Our cells leave out the chapter's `raise_for_status()` and `timeout=`. If the server stopped answering, as it did after 13 to 17 requests, what would the cell do?
- Asked for a folder that doesn't exist, the portal still answers 200 with JSON, and `folder["path"]` is `None`; the reason is in `folder["errMsg"]`. What would the given loop do next?

</details>

## Step 7 — Text three ways (9 minutes)

The minutes of December 5, 2024 were written in Word and saved as PDF, so every page has characters. pypdf writes them in the order the page draws them; its `extraction_mode="layout"` places each piece by its position instead. [pdfplumber](https://github.com/jsvine/pdfplumber#extracting-text) starts from where each character ends up, joining characters into words when the gap is under 3 points, and words into lines when their tops line up. Print the top of page 1 all three ways, and compare. See [Text with pypdf](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#text-with-pypdf).

In [ ]:
# minutes_2024: PdfReader(get_saved_copy("council-minutes-2024-12-05.pdf"))
# Your code here

In [ ]:
# plain_text: page 1's extract_text(); print its first 300 characters
# Your code here

In [ ]:
# layout_text: page 1's extract_text(extraction_mode="layout"); print its first 600 characters
# Your code here

In [ ]:
# minutes_pdf: pdfplumber.open() the same file; plumber_text: its pages[0].extract_text(); print 300
# Your code here

### ⏱ Microsprint (4 minutes, alone or in pairs)

A short version of Additional Exercise 4, on a table instead of prose.

**Do:** set `look_for` to `"Total Sales"`, and run the given cell: it reads the report's summary page all three ways, and prints the lines that hold `look_for`. Then try `"economic activity"`.
**Share:** whether the three agree on these rows, and what a pattern that reads one row per line would take as the subtotal's label, "Total Sales & Use Tax based on current economic activity", which wraps onto two lines on the page.

In [ ]:
# Microsprint: look_for: "Total Sales"
# Your code here

In [ ]:
# Given: the report's summary page three ways; each line that holds look_for, with runs of spaces shrunk
summary_plain = PdfReader(report_2024).pages[1].extract_text()
summary_layout = PdfReader(report_2024).pages[1].extract_text(extraction_mode="layout")
summary_plumber = summary_page.extract_text()
for line in summary_plain.splitlines():
    if look_for in line:
        print("pypdf, plain: ", " ".join(line.split()))
for line in summary_layout.splitlines():
    if look_for in line:
        print("pypdf, layout:", " ".join(line.split()))
for line in summary_plumber.splitlines():
    if look_for in line:
        print("pdfplumber:   ", line)

<details>
<summary>💬 Discuss</summary>

- pypdf's plain mode broke "Mayor" into "M" and "ayor". Is it broken in the layout text, or in pdfplumber's? What would you check before relying on any one of them for a new file?
- pypdf printed "P age 1" first, though the page number sits at the bottom. What does that tell you about the order a PDF is drawn in?
- In the microsprint, where did the subtotal's figures land: on its label's first line, or its second? What would a one-row-per-line pattern call that row?

</details>

## Step 8 — Repair the text, then find things in it (11 minutes)

Repair what pypdf broke, matching what the extraction produced rather than what the page shows: a search for "Page 1" misses "P age 1". Make one repair by hand with [`str.replace()`](https://docs.python.org/3/library/stdtypes.html#str.replace). A given cell then joins all eight pages and makes the chapter's four repairs with [`re.sub()`](https://docs.python.org/3/library/re.html#re.sub), using *regular expressions*: patterns that match many strings at once. Count one word by hand, then read the chapter's five patterns, each run with [`re.findall()`](https://docs.python.org/3/library/re.html#re.findall). See [Cleanup Strategies](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#cleanup-strategies).

In [ ]:
# fixed_text: plain_text with "P age 1" replaced by ""; is "P age 1" in each of them?
# Your code here

In [ ]:
# Given: every page's text, joined, then the chapter's four repairs, one re.sub() each.
# In a pattern, \d is any digit, + is "one or more", [ \t] is a space or a tab, and ^ and $ a line's ends
page_texts = []
for page in minutes_2024.pages:
    page_texts.append(page.extract_text())
raw_text = "\n".join(page_texts)
full_text = re.sub(r"^P ?age \d+[ \t]*$", "", raw_text, flags=re.MULTILINE)  # the page number, "P age 1"
full_text = re.sub(r"\b([A-Z]{1,2})\n(?=[A-Za-z])", r"\1", full_text)        # "M" and "ayor" become "Mayor"
full_text = re.sub(r"[ \t]+", " ", full_text)                                 # spaces and tabs, but not newlines
full_text = re.sub(r" *\n\s*", "\n", full_text).strip()                       # strip each line; drop empty ones
print(full_text[:300])

In [ ]:
# How many times do the minutes say "Carried"? Print full_text.count("Carried")
# Your code here

In [ ]:
# Given: the chapter's five patterns, each run with re.findall(), which returns every match in a list.
# MONTH matches any month's name: | means "or"
MONTH = r"(?:January|February|March|April|May|June|July|August|September|October|November|December)"
print("dates in words:", re.findall(MONTH + r" \d{1,2}, \d{4}", full_text))
print("dates with slashes:", re.findall(r"\d{1,2}/\d{1,2}/\d{2,4}", full_text))
print("clock times:", len(re.findall(r"\d{1,2}:\d{2} [ap]\.m\.", full_text)))
print("tallies:", re.findall(r"(Carried|Failed) (\d+):(\d+)", full_text))
print("dollars:", re.findall(r"\$[\d,]+(?:\.\d{2})?", full_text))

### ⏱ Microsprint (5 minutes, alone or in pairs)

[The chapter's advice](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#cleanup-strategies), "check a repair written for one file on the next, and print what it changed", cut to five minutes.

**Do:** set `check_file` to `"council-minutes-2023-12-07.pdf"`, and run the given cell. It counts the places on each page where the second repair would join two lines, and prints the suspicious ones: a join whose capitals follow a character that is neither a letter nor a space, as in "A-F". (The other joins include the first names of members of the public who spoke, so it prints counts, not words.)
**Share:** how many joins, how many look suspicious, and which of those is really wrong.

In [ ]:
# Microsprint: check_file: "council-minutes-2023-12-07.pdf"
# Your code here

In [ ]:
# Given: on each page of check_file, count the second repair's joins, and print the suspicious ones
joins = 0
number = 0
for page in PdfReader(get_saved_copy(check_file)).pages:
    number = number + 1
    page_text = page.extract_text()
    joins = joins + len(re.findall(r"\b[A-Z]{1,2}\n(?=[A-Za-z])", page_text))
    for match in re.findall(r"\S*[^\w\s][A-Z]{1,2}\n\S+", page_text):
        print("page", number, ":", match.replace("\n", ""))
print(joins, "joins in all")

<details>
<summary>💬 Discuss</summary>

- The join repair was right every time on the December 5 minutes, and wrong at least once in the microsprint. When has a repair been tested enough to run on a hundred files?
- `count("Carried")` and the tally pattern both found 2. The minutes also say "The motion carried with a vote of 9:0", with a small c. What did each of them miss, and how would you find out how many votes there were?
- Week 7 dropped stopwords before counting words. What else would a count of a topic word like "housing" need before you compared one meeting with another?

</details>

## Step 9 — Tables with pdfplumber (15 minutes)

pdfplumber's [table finder](https://github.com/jsvine/pdfplumber#extracting-tables) rebuilds a grid from the page's rules and the sides of its rectangles, and makes a cell wherever rules close a box. Ask it for the summary page's tables, and compare the first table's last row with the page. A table's `bbox`, its *bounding box*, is `(x0, top, x1, bottom)`, in points. The fix, given: the column edges yourself, as [`explicit_vertical_lines`](https://github.com/jsvine/pdfplumber#table-extraction-settings), and a crop that ends below the totals row. See [Finding Tables](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#finding-tables).

In [ ]:
# tables: summary_page.find_tables(); print how many tables it found
# Your code here

In [ ]:
# default_rows: tables[0].extract(), a list of rows; print its last row
# Your code here

In [ ]:
# Print where the table ends, tables[0].bbox; then, as on Monday, where the totals row starts:
# found: summary_page.search("Total Sales and Use Tax"); print found[0]["top"]
# Your code here

In [ ]:
# Given: the left and right edge of every cell the finder did find, each once, in order.
# A set keeps one of each value, and sorted() makes it a list, smallest first
first_table = tables[0]
edges = set()
for cell in first_table.cells:
    edges.add(round(cell[0], 2))  # the cell's left edge
    edges.add(round(cell[2], 2))  # and its right edge
edges = sorted(edges)
print(edges)

In [ ]:
# Given: the settings: columns at the edges we give ("explicit"), and rows from the page's rules ("lines")
settings = {"vertical_strategy": "explicit",
            "explicit_vertical_lines": edges,
            "horizontal_strategy": "lines"}

In [ ]:
# table_bottom: 395, in points: below the totals row, and above the footnote under it
# Your code here

In [ ]:
# Given: crop the page to the table's area, down to table_bottom, and read the table with settings
area = summary_page.crop((first_table.bbox[0], first_table.bbox[1], first_table.bbox[2], table_bottom))
fixed_rows = area.extract_table(settings)
print(len(default_rows), "rows before,", len(fixed_rows), "after")
print(fixed_rows[-1])

In [ ]:
# Given: a DataFrame of the label and both years' figures; blank cells become pd.NA, and dropna() drops
# those rows; then each label on one line, and the figures as numbers, without "$" and ","
summary_df = pd.DataFrame(fixed_rows).iloc[:, [1, 3, 4]]
summary_df.columns = ["source", "ytd_2023", "ytd_2024"]
summary_df = summary_df.replace("", pd.NA).dropna()
summary_df["source"] = summary_df["source"].str.replace("\n", " ")
summary_df["ytd_2023"] = pd.to_numeric(summary_df["ytd_2023"].str.replace(r"[$,]", "", regex=True))
summary_df["ytd_2024"] = pd.to_numeric(summary_df["ytd_2024"].str.replace(r"[$,]", "", regex=True))
summary_df

In [ ]:
# Given: save the table as Wednesday's checkpoint, for Friday
summary_df.to_csv("summary.csv", index=False)

In [ ]:
# Given, only if the cells above stopped: remove the "#" from the next two lines,
# and run this cell. It loads the table the key's run saved on 2026-11-17
# data_base = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-09/data/"
# summary_df = pd.read_csv(data_base + "summary.csv")

### ⏱ Microsprint (5 minutes, alone or in pairs)

A check of Step 9's finding on another year, after Additional Exercise 2. The chapter shows the totals row falling outside the table in December 2024's report, and says the other reports' tables have different rules.

**Do:** set `other_report` to another year-end report, `"revenue-report-2018.pdf"`, `-2019.pdf`, `-2021-12.pdf`, `-2022-12.pdf`, or `-2023-12.pdf` (2020's summary is on page 3, so skip it), and run the given cell.
**Share:** your year, where the table ends and where the totals row starts, and the label of the table's last row.

In [ ]:
# Microsprint: other_report: "revenue-report-2018.pdf", or another year's
# Your code here

In [ ]:
# Given: other_report's summary page: where its first table ends, where its totals row starts,
# and the first cell of the table's last row
other_page = pdfplumber.open(get_saved_copy(other_report)).pages[1]
other_table = other_page.find_tables()[0]
other_found = other_page.search("Total Sales and Use Tax")
print("the table ends at", round(other_table.bbox[3]), "| the totals row starts at", round(other_found[0]["top"]))
print("last row:", other_table.extract()[-1][0])

<details>
<summary>💬 Discuss</summary>

- The default finder dropped the totals row and raised nothing. What told us, and what would have told us if we hadn't looked at the page?
- Did the finder drop the totals row in your year too? What does the class's answer say about checking one file and trusting the rest?
- Typed as `table_bottom = 900`, the crop stops with a `ValueError`: the box isn't on the page. Typed as 385, it raises nothing. Which check in Step 10 would notice what it lost?
- Week 6's `pd.read_html()` turned an HTML table into a DataFrame in one call. What does HTML's `<td>` give you for free that a PDF table doesn't?

</details>

## Step 10 — Check the extractions (7 minutes)

Extraction fails without errors, so check it, cheapest first. **Count** the rows: the page shows nine. Check the **sums**: the five sources add up to the subtotal, and the subtotal and the two audit rows to the total, within a few dollars of rounding. Then **a sample coded by hand**: who was present at the December 5, 2024 roll call, as [precision and recall](https://en.wikipedia.org/wiki/Precision_and_recall) against the members page 1 names. A *set*, written in braces, holds each value once. See [Checking an Extraction](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#checking-an-extraction).

In [ ]:
# Count: print len(summary_df), beside the page's nine rows of figures
# Your code here

In [ ]:
# Given: the sums, for both years at once. .loc[] picks a row by its label
figures = summary_df.set_index("source")
subtotal = figures.loc["Total Sales & Use Tax based on current economic activity"]
audits = figures.loc["Sales Tax Audit Revenue"] + figures.loc["Use Tax Audit Revenue"]
print("sources minus subtotal:", (figures.iloc[:5].sum() - subtotal).to_dict())
print("subtotal plus audits minus total:", (subtotal + audits - figures.loc["Total Sales and Use Tax"]).to_dict())

In [ ]:
# present_by_hand: a set of the nine members page 1 names as present: "Adams", "Benjamin", "Brockett",
# "Folkerts", "Marquis", "Schuchard", "Speer", "Wallach", and "Winer"
# Your code here

In [ ]:
# Given: the extraction to check: the capitalized words after "present:" on its line of plain_text.
# & keeps what two sets share, and - what one has that the other doesn't
present_line = re.search(r"present:(.*)$", plain_text, re.MULTILINE)
present_found = set(re.findall(r"[A-Z][a-z]+", present_line.group(1)))
correct = present_found & present_by_hand
print("precision:", round(len(correct) / len(present_found), 2), "| recall:", round(len(correct) / len(present_by_hand), 2))
print("missed:", sorted(present_by_hand - present_found))

<details>
<summary>💬 Discuss</summary>

- Precision was perfect and recall wasn't. Which of the two would a quick look at the output have shown you, and why?
- The sums say we copied the city's figures faithfully. Why don't they say the figures are right?

</details>

## End of Part 2

On Friday, Part 3 reads a picture of a table with OCR, one row first, and then reads the totals row of seven reports into one time series. It starts with its own setup step, and loads today's table from the course's repository.

# Part 3 — Friday: OCR, and seven reports

Today: a page with no text layer, and a question across years. Exhibit 3, the last page of the December 2024 report, is a picture of a table: we read one row of it with OCR, then the whole picture, and check what came back. Then we read one row by position from all seven year-end reports, and plot eight years of sales and use tax.

## Step 11 — Set up again (3 minutes)

Part 3 starts fresh. Put your email address in the User-Agent, and run the two given cells. The second holds `get_saved_copy()`, and loads Wednesday's checkpoint, `summary.csv`, the summary table we fixed, from the course's repository.

In [ ]:
# Given: Part 3's setup. Put your email address in HEADERS, then run it
import re
import time
import requests
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import pdfplumber
import pytesseract
from pypdf import PdfReader

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
BOOK_DATA = "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-09/"
PDF_DIR = Path("pdfs")
PDF_DIR.mkdir(exist_ok=True)

In [ ]:
# Given: get_saved_copy(), the December 2024 report, and Wednesday's table, which the key's run saved
# on 2026-11-17
def get_saved_copy(name):
    path = PDF_DIR / name
    if not path.exists():
        response = requests.get(BOOK_DATA + name, headers=HEADERS)
        path.write_bytes(response.content)
        time.sleep(1)
    return path

report_2024 = get_saved_copy("revenue-report-2024-12.pdf")
data_base = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-09/data/"
summary_df = pd.read_csv(data_base + "summary.csv")
summary_df

<details>
<summary>💬 Discuss</summary>

- `summary.csv` is a saved copy of what Wednesday's cells read. The PDF hasn't changed, so what can the saved copy not tell you that rerunning Wednesday's cells can?

</details>

## Step 12 — OCR: one row first, then the picture (16 minutes)

pypdf hands you a page's picture as `page.images[0].image`, a Pillow image, and [pytesseract](https://github.com/madmaze/pytesseract#usage) runs Tesseract on it. The given cell turns it gray ("L"), which suits OCR, and crops one row, 2021's, by its box in pixels. Read that row's figures off the image, then compare. Then read the whole picture with `config="--psm 6"`, the [page segmentation mode](https://tesseract-ocr.github.io/tessdoc/ImproveQuality.html#page-segmentation-method) for one uniform block of text. See [When There Is No Text Layer: OCR](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#when-there-is-no-text-layer-ocr).

In [ ]:
# exhibit_page: page 15 of report_2024, PdfReader(report_2024).pages[14]; how many images does it have?
# Your code here

In [ ]:
# exhibit_image: its first image's .image; print its .size, in pixels (width, height)
# Your code here

In [ ]:
# Given: the image in gray, and one row of it, 2021's: a crop of (left, top, right, bottom), in pixels
exhibit_3 = exhibit_image.convert("L")
row_2021 = exhibit_3.crop((0, 151, 1000, 180))
row_2021

In [ ]:
# Print pytesseract.image_to_string(row_2021): does every figure match the image?
# Your code here

In [ ]:
# ocr_text: image_to_string() of all of exhibit_3, with config="--psm 6"
# Your code here

In [ ]:
# ocr_lines: ocr_text.splitlines(); print how many lines, and line 5, 2021's
# Your code here

In [ ]:
# Given: the chapter's patterns. NUMBER is a figure as OCR may read it, with a stray dash, or periods
# for commas. ROW is a line of the table: a category's name or nothing, a year, then 13 figures
NUMBER = r"[—–-]?\d[\d.,]*"
ROW = re.compile(r"^(?P<label>[A-Za-z ]*?)\s*(?P<year>20\d\d)\s+(?P<figures>(?:" + NUMBER + r"\s+){13})")

In [ ]:
# Given: each line ROW matches becomes a row of the label, the year, and the figures' digits only,
# since the table's dollars are whole; a line with a year that doesn't match is set aside
ocr_rows = []
for line in ocr_lines:
    match = ROW.match(line)
    if match:
        figures = re.sub(r"[^\d\s]", "", match["figures"]).split()
        ocr_rows.append([match["label"].strip(), match["year"]] + figures)
    elif re.search(r"\b20\d\d\b", line):
        print("set aside:", " ".join(line.split()[:6]))
print(len(ocr_rows), "of the table's 30 rows parsed")

In [ ]:
# Given: the check. Each category's name starts only its first row, so ffill() fills it down;
# then each row's twelve months should add up to its total
ocr_df = pd.DataFrame(ocr_rows).rename(columns={0: "category", 1: "year", 14: "total"})
ocr_df["category"] = ocr_df["category"].replace("", pd.NA).ffill()
months = ocr_df.iloc[:, 2:14].astype(int).sum(axis=1)
gaps = (months - ocr_df["total"].astype(int)).abs()
print("largest gap between the months and the total:", gaps.max(), "dollars")

In [ ]:
# Given: two documents should agree: OCR's 2024 total, from the picture, and Wednesday's table,
# from the summary page's text
ocr_total = int(ocr_df[(ocr_df["category"] == "TOTAL") & (ocr_df["year"] == "2024")]["total"].iloc[0])
text_total = summary_df.set_index("source").loc["Total Sales and Use Tax", "ytd_2024"]

In [ ]:
# Print ocr_total and text_total: do they match?
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

Change one parameter: the page segmentation mode.

**Do:** set `psm_choice` to `"4"` (one column of text of varying sizes) or `"3"` (Tesseract's default, which finds the layout itself), and run the given cell. It reads `exhibit_3` again, counts the lines that `ROW` matches, and times it.
**Share:** your mode, its number of lines and of rows parsed, and how many seconds it took.

In [ ]:
# Microsprint: psm_choice: "4", or "3"
# Your code here

In [ ]:
# Given: OCR exhibit_3 with psm_choice; count the lines ROW matches, and time it
started = time.time()
psm_lines = pytesseract.image_to_string(exhibit_3, config="--psm " + psm_choice).splitlines()
parsed = 0
for line in psm_lines:
    if ROW.match(line):
        parsed = parsed + 1
print("--psm", psm_choice, ":", len(psm_lines), "lines,", parsed, "rows parsed,", round(time.time() - started, 1), "seconds")

In [ ]:
# Given: a scan with a text layer. Page 7 of the December 4, 2018 minutes is one image, and the
# scanner's OCR wrote its motion as "CouNcrr, Mnunrn WnlvnR MovED To ADopr". Tesseract reads it again
scan = PdfReader(get_saved_copy("council-minutes-2018-12-04.pdf")).pages[6].images[0].image
for line in pytesseract.image_to_string(scan).splitlines():
    if "MOVED" in line or "PASSED" in line:
        print(line)

<details>
<summary>💬 Discuss</summary>

- On one row, did Tesseract read every figure the way you read it off the image? On the whole picture, line 5 is the same row. What changed, and why would a row's sum catch it?
- Every parsed row adds up within a few dollars. Name an OCR error that a row's sum can't catch.
- Chapter 9 calls a picture of a table the extreme case of soft enclosure. In week 3's terms, who is fenced out by it, and what does it cost them to get back in?

</details>

## Step 13 — Seven reports, one time series (12 minutes)

The chapter's case study: total sales and use tax from seven year-end reports. Their tables' rules differ, so read the one row by position. By hand, on December 2024's: [`search()`](https://github.com/jsvine/pdfplumber#extracting-text) finds the label; a [`crop()`](https://github.com/jsvine/pdfplumber#the-pdfplumberpage-class) from its right end to the page's edge, as tall as its line, holds the figures; `split()` makes a list. The given loops find each report's summary page by its first sentence (2020's is page 3), and read every row. See [Case Study: Boulder Revenue Reports](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#case-study-boulder-revenue-reports).

In [ ]:
# summary_page: page 2 of pdfplumber.open(report_2024); does its text hold "This report provides"?
# Your code here

In [ ]:
# label: the first match of summary_page.search("Total Sales and Use Tax"); print its "x1"
# Your code here

In [ ]:
# box: from 1 point right of label["x1"] to the page's right edge, 1 point above and below the label
# Your code here

In [ ]:
# row: summary_page.crop(box); row_text: its extract_text(); print row_text.split()
# Your code here

In [ ]:
# Given: the seven year-end reports, oldest first, and the page number of each one's summary,
# the page among its first four whose text holds "This report provides"
REPORTS = ["revenue-report-2018.pdf", "revenue-report-2019.pdf", "revenue-report-2020.pdf",
           "revenue-report-2021-12.pdf", "revenue-report-2022-12.pdf",
           "revenue-report-2023-12.pdf", "revenue-report-2024-12.pdf"]
summary_pages = {}
for name in REPORTS:
    for page in pdfplumber.open(get_saved_copy(name)).pages[:4]:
        if "This report provides" in page.extract_text():
            summary_pages[name] = page.page_number
print(summary_pages)

In [ ]:
# Given: our four typed lines, for every report. The heading's two years come from a search() with
# two groups, and the row's first two figures are those two years' totals
totals = []
for name in REPORTS:
    page = pdfplumber.open(get_saved_copy(name)).pages[summary_pages[name] - 1]
    years = page.search(r"(20\d\d) +(20\d\d)")[0]["groups"]
    found = page.search("Total Sales and Use Tax")[0]
    row_box = (found["x1"] + 1, found["top"] - 1, page.width, found["bottom"] + 1)
    figures = page.crop(row_box).extract_text().split()
    totals.append({"report": name, "year": int(years[0]), "total": figures[0]})
    totals.append({"report": name, "year": int(years[1]), "total": figures[1]})
totals = pd.DataFrame(totals)
totals["total"] = pd.to_numeric(totals["total"].str.replace(r"[$,]", "", regex=True))
totals.to_csv("totals.csv", index=False)

In [ ]:
# Given, only if the cell above stopped: remove the "#" from the next line, and run this cell.
# It loads the totals the key's run saved on 2026-11-17 (data_base is from Step 11)
# totals = pd.read_csv(data_base + "totals.csv")

In [ ]:
# Given: the years printed twice, in their own report and as "the year before" in the next;
# the smallest and largest printing, and how far apart they are
printings = totals.groupby("year")["total"].agg(["count", "min", "max"])
twice = printings[printings["count"] == 2]
twice.assign(difference=twice["max"] - twice["min"])

In [ ]:
# Given: each year's latest printing (REPORTS runs oldest to newest), the percent change from the year
# before, and a bar chart
latest = totals.groupby("year").last()
latest["change_pct"] = (latest["total"].pct_change() * 100).round(2)
print(latest[["total", "change_pct"]].to_string())
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(latest.index, latest["total"] / 1e6)
ax.set_title("Boulder's sales and use tax by year, 2017–2024 (each year's latest printing)")
ax.set_xlabel("Year")
ax.set_ylabel("Total collected (millions of dollars)")
plt.show()

<details>
<summary>💬 Discuss</summary>

- Sales and use tax fell in 2020 and rose in 2021. What would you need before calling that a pandemic effect?
- One year is printed two ways, in two reports. Which printing would you use, and what would you write in the data's notes?
- Without the `+ 1` in `box`, the crop starts on the label's last letter, and the row's first word is `'x'`. Which given cell would stop, and would its error tell you why?
- Week 3 named oversight as a value for public-interest data work. What can someone outside City Hall check with this series that they couldn't with the PDFs alone?

</details>

## Step 14 — What did we learn? (4 minutes)

In a few sentences, as comments or in a Markdown cell:

- Which pages we opened this week needed OCR, and how could you tell without looking at them?
- Name one extraction that failed without an error this week, and the check that caught it.
- How did Boulder's sales and use tax change from 2019 to 2024, and what lets you trust the figures enough to put eight years side by side?
- For a hundred council minutes, would you start from pypdf with repairs or from pdfplumber? What would you check first?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Sunday):

- Step 1, collect three PDFs from one government source, politely, into `pdfs/`: Step 1's download by hand (User-Agent, look first, save the bytes) and its given `get_saved_copy()` (download once, pause after each), and Step 6's `%PDF-` check.
- Step 2, open one and print its page count and page 1's raw text: Step 2's `PdfReader`, and Step 7's three ways.
- Step 3, write a cleanup function and show a passage before and after: Step 8's repair by hand and the four given repairs, and its microsprint's check on a second file.
- Step 4, extract one element with `re.findall()`: Step 8's five given patterns.
- Step 5, loop the pipeline over all three PDFs into a DataFrame: Step 4's given loop over three files, and Step 13's over seven reports.
- Step 6, plot your element's count per document, and check the pages with little text: Step 13's plot, and Step 4's characters and images on every page.
- Step 7, interpret, and judge extraction quality: the checks of Steps 10 and 12.

In [ ]:
# Your answers here, as comments


## Friday: from the lab to the textbook (15 minutes)

What did the lab turn up that the chapter should know? Some places to look:

- **Outputs that changed.** In Step 7, pypdf's layout mode kept "Council Chambers" whole, where the chapter prints "C ham bers" and builds a paragraph on it. With Tesseract 5.3.4, Step 12 parsed 29 of Exhibit 3's rows and set 1 aside; the chapter, with 5.5.3, reports 28 and 2. What did yours print?
- **A microsprint that would make a good exercise.** Step 9's microsprint asked whether the default table finder drops the totals row in other years too. Is the answer worth a sentence in "Finding Tables", or an Additional Exercise?
- **Code the lab wrote differently.** The lab wrote the chapter's `page_profile()`, `clean_page_text()`, `summary_frame()`, `check_summary()`, and the case study's three functions as straight lines, typed for one case and given for the rest. Which read more easily that way, and where did you miss the function?
- **A sentence that confused you.** Did any of the chapter's explanations of pypdf's two modes, the join rule, or the table settings leave you guessing during the lab?
- **A check the chapter could add.** Step 8 showed the tally pattern finds votes only in the motion tables. Is there a short check that would count what it misses?

[`handouts/common/revision-framework.md`](../../handouts/common/revision-framework.md) says how to propose a revision.